In [2]:
# ================================================================
# AMPORA PS 26052
# HIGH-BAND AI / ML MODULE
# ================================================================
#
# IMPORTANT:
# This code implements ONLY:
#
#        1.5 kHz - 8 kHz HIGH-BAND
#                    ↓
#                  DC-CRN
#                    ↓
#              Enhanced HIGH-BAND
#
# The LOW BAND (80 Hz - 1.5 kHz) NLMS branch is NOT implemented.
# It is a separate branch in the complete AMPORA hardware system.
#
# ================================================================


# ================================================================
# 0. INSTALL DEPENDENCIES
# ================================================================

!pip -q install soundfile scipy librosa pystoi pesq tqdm pandas onnx onnxruntime


# ================================================================
# 1. IMPORTS
# ================================================================

import os
import re
import math
import random
import zipfile
import subprocess
import json

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import soundfile as sf

from scipy import signal
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ================================================================
# 2. CONFIGURATION
# ================================================================

SEED = 42

# Audio
SR = 16000

# AMPORA high-band limits
GLOBAL_LOW = 80.0
CROSSOVER = 1500.0
GLOBAL_HIGH = 8000.0

# STFT
NFFT = 512
WIN_LENGTH = 320
HOP_LENGTH = 128

# Training
EPOCHS = 20
BATCH_SIZE = 8

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4

GRAD_CLIP = 5.0

# Audio segment
SEGMENT_SECONDS = 1.28
SEGMENT_SAMPLES = int(SR * SEGMENT_SECONDS)

# Training SNR
SNR_MIN = -15
SNR_MAX = 15

# Dataset
DATASET_ROOT = "/content/AMPORA_V3_HARD_REALVOICE_DATASET"
ZIP_PATH = None

# Optional FSDD
USE_FSDD = True

# Optional public speech/noise datasets can be added under DATASET_ROOT.
# The training code also generates synthetic defence-style noise so the model
# does not depend on one public noise corpus.

FSDD_ROOT = "/content/free-spoken-digit-dataset"

FSDD_URL = (
    "https://github.com/Jakobovski/"
    "free-spoken-digit-dataset.git"
)

# ================================================================
# GOOGLE DRIVE CHECKPOINT STORAGE
# ================================================================
# Checkpoints are stored in Drive so a Colab GPU disconnect does NOT
# erase completed epochs. You will be asked to authorize Drive once.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER"
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Checkpoint directory:", CHECKPOINT_DIR)


# ================================================================
# 3. DEVICE + SEED
# ================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


def seed_everything(seed=SEED):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


seed_everything()


# ================================================================
# 4. UPLOAD + EXTRACT V3 DATASET
# ================================================================

from google.colab import files

def upload_and_extract_dataset():
    global ZIP_PATH

    existing_wavs = list(Path(DATASET_ROOT).rglob("*.wav")) if os.path.isdir(DATASET_ROOT) else []
    if existing_wavs:
        print(f"Dataset already extracted: {len(existing_wavs)} WAV files found.")
        return

    print("=" * 64)
    print("UPLOAD AMPORA V3 DATASET ZIP")
    print("=" * 64)
    print("Select your AMPORA_V3_HARD_REALVOICE_DATASET.zip file.")
    print("The code automatically detects the ZIP filename.")
    print()

    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("No file was uploaded. Please upload the V3 dataset ZIP.")

    zip_candidates = [name for name in uploaded.keys() if name.lower().endswith(".zip")]

    if not zip_candidates:
        raise RuntimeError(
            "No ZIP file was uploaded. Please upload "
            "AMPORA_V3_HARD_REALVOICE_DATASET.zip."
        )

    preferred = [
        name for name in zip_candidates
        if "AMPORA_V3_HARD_REALVOICE_DATASET".lower() in name.lower()
    ]

    selected_name = preferred[0] if preferred else zip_candidates[0]
    ZIP_PATH = os.path.join("/content", selected_name)

    print("Dataset ZIP detected:", ZIP_PATH)

    if not zipfile.is_zipfile(ZIP_PATH):
        raise RuntimeError(f"Invalid ZIP file: {ZIP_PATH}")

    print("Extracting dataset...")
    os.makedirs(DATASET_ROOT, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATASET_ROOT)

    wavs = list(Path(DATASET_ROOT).rglob("*.wav"))

    if not wavs:
        raise RuntimeError(
            "ZIP extracted successfully, but no WAV files were found. "
            "Please check that you uploaded the AMPORA V3 dataset."
        )

    print(f"Extraction completed. WAV files found: {len(wavs)}")

upload_and_extract_dataset()

# ================================================================
# 5. OPTIONAL FSDD
# ================================================================

def download_fsdd():

    if not USE_FSDD:

        return None

    recordings = os.path.join(
        FSDD_ROOT,
        "recordings"
    )

    if os.path.isdir(recordings):

        print("FSDD already available.")

        return FSDD_ROOT

    print("Downloading FSDD...")

    try:

        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                FSDD_URL,
                FSDD_ROOT
            ],
            check=True
        )

        print("FSDD downloaded.")

        return FSDD_ROOT

    except Exception as e:

        print(
            "FSDD download failed:",
            e
        )

        return None


FSDD_ROOT_ACTUAL = download_fsdd()


# ================================================================
# 6. AUDIO FUNCTIONS
# ================================================================

def read_audio(path):

    audio, fs = sf.read(
        path,
        dtype="float32"
    )

    audio = np.asarray(
        audio,
        dtype=np.float32
    )

    # Stereo → mono
    if audio.ndim == 2:

        audio = np.mean(
            audio,
            axis=1
        )

    # Resample
    if fs != SR:

        g = math.gcd(
            int(fs),
            SR
        )

        up = SR // g
        down = int(fs) // g

        audio = signal.resample_poly(
            audio,
            up,
            down
        ).astype(np.float32)

    audio = np.nan_to_num(audio)

    peak = np.max(
        np.abs(audio)
    ) + 1e-8

    if peak > 1.0:

        audio = audio / peak

    return audio.astype(
        np.float32
    )


def rms(x):

    return float(
        np.sqrt(
            np.mean(x * x)
            + 1e-12
        )
    )


def fit_length(
    x,
    length
):

    if len(x) >= length:

        if len(x) == length:

            return x.copy()

        start = random.randint(
            0,
            len(x) - length
        )

        return x[
            start:start + length
        ].copy()

    repeats = int(
        np.ceil(
            length /
            max(1, len(x))
        )
    )

    y = np.tile(
        x,
        repeats
    )

    return y[
        :length
    ].copy()


def normalize_peak(
    x,
    peak=0.98
):

    current = (
        np.max(
            np.abs(x)
        )
        + 1e-8
    )

    gain = min(
        1.0,
        peak / current
    )

    return (
        x * gain
    ).astype(
        np.float32
    )


def scale_noise_to_snr(
    clean,
    noise,
    snr_db
):

    clean_rms = rms(clean)
    noise_rms = rms(noise)

    if noise_rms < 1e-8:

        return np.zeros_like(
            noise
        )

    desired_noise_rms = (
        clean_rms /
        (10 ** (snr_db / 20))
    )

    gain = (
        desired_noise_rms /
        (noise_rms + 1e-8)
    )

    return (
        noise * gain
    ).astype(
        np.float32
    )


# ================================================================
# 7. AMPORA HIGH-BAND EXTRACTION
# ================================================================
#
# Common front-end:
#
#       80 Hz ---------------- 8 kHz
#
# Then high-band branch:
#
#       1.5 kHz --------------- 8 kHz
#
# ONLY THIS SIGNAL ENTERS DC-CRN.
# ================================================================


def global_bandpass(x, low=80, high=8000, order=4):
    """Safe Butterworth band-pass; upper cutoff stays below Nyquist."""
    x = np.asarray(x, dtype=np.float32)
    nyquist = SR / 2.0
    safe_low = max(float(low), 1.0)
    safe_high = min(float(high), nyquist - 10.0)
    if safe_low >= safe_high:
        raise ValueError(f"Invalid bandpass: {safe_low} Hz - {safe_high} Hz")
    sos = signal.butter(order, [safe_low, safe_high], btype="bandpass", fs=SR, output="sos")
    try:
        y = signal.sosfiltfilt(sos, x)
    except ValueError:
        y = signal.sosfilt(sos, x)
    return np.asarray(y, dtype=np.float32)


def high_crossover(
    x,
    cutoff=1500,
    order=2
):

    sos = signal.butter(
        order,
        cutoff,
        btype="highpass",
        fs=SR,
        output="sos"
    )

    return signal.sosfilt(
        sos,
        x
    ).astype(
        np.float32
    )


def extract_high_band(x):

    # Step 1:
    # Common 80 Hz - 8 kHz front end

    x = global_bandpass(
        x,
        80,
        8000
    )

    # Step 2:
    # High output of ~1.5 kHz crossover

    x = high_crossover(
        x,
        1500
    )

    return x


# ================================================================
# 8. SYNTHETIC HARD NOISE GENERATORS
# ================================================================

def lowpass(
    x,
    cutoff,
    order=4
):

    sos = signal.butter(
        order,
        cutoff,
        btype="lowpass",
        fs=SR,
        output="sos"
    )

    return signal.sosfilt(
        sos,
        x
    ).astype(np.float32)


def bandpass(
    x,
    low,
    high,
    order=3
):

    sos = signal.butter(
        order,
        [
            low,
            high
        ],
        btype="bandpass",
        fs=SR,
        output="sos"
    )

    return signal.sosfilt(
        sos,
        x
    ).astype(np.float32)


def white_noise(n):

    return np.random.randn(
        n
    ).astype(
        np.float32
    )


def pink_noise(n):

    freqs = np.fft.rfftfreq(
        n,
        1 / SR
    )

    amplitude = np.ones_like(
        freqs
    )

    amplitude[1:] = (
        1 /
        np.sqrt(freqs[1:])
    )

    phase = np.exp(
        1j *
        2 *
        np.pi *
        np.random.rand(
            len(freqs)
        )
    )

    spectrum = (
        amplitude *
        phase
    )

    y = np.fft.irfft(
        spectrum,
        n=n
    )

    y /= (
        np.std(y)
        + 1e-8
    )

    return y.astype(
        np.float32
    )


def engine_noise(n):

    t = np.arange(
        n
    ) / SR

    f = np.random.uniform(
        70,
        160
    )

    y = (
        0.9 *
        np.sin(
            2 *
            np.pi *
            f *
            t
        )
    )

    y += (
        0.45 *
        np.sin(
            2 *
            np.pi *
            2*f *
            t
        )
    )

    y += (
        0.25 *
        np.sin(
            2 *
            np.pi *
            3*f *
            t
        )
    )

    y += (
        0.2 *
        lowpass(
            np.random.randn(n)
            .astype(np.float32),
            500
        )
    )

    return y.astype(
        np.float32
    )


def rotor_noise(n):

    t = np.arange(
        n
    ) / SR

    base = np.random.uniform(
        18,
        45
    )

    y = np.zeros(
        n,
        dtype=np.float32
    )

    for k in range(
        1,
        13
    ):

        y += (
            1 /
            (k ** 0.85)
        ) * np.sin(
            2 *
            np.pi *
            base *
            k *
            t
        )

    modulation = (
        0.55 +
        0.45 *
        np.sin(
            2 *
            np.pi *
            np.random.uniform(
                1,
                4
            ) *
            t
        )
    )

    y *= modulation

    y += (
        0.15 *
        lowpass(
            np.random.randn(n)
            .astype(np.float32),
            900
        )
    )

    return y.astype(
        np.float32
    )


def wind_noise(n):

    return lowpass(
        pink_noise(n),
        np.random.uniform(
            500,
            1800
        ),
        3
    )


def radio_noise(n):

    t = np.arange(
        n
    ) / SR

    background = bandpass(
        np.random.randn(n)
        .astype(np.float32),
        2500,
        7600
    )

    carrier_frequency = (
        np.random.uniform(
            1800,
            6500
        )
    )

    carrier = np.sin(
        2 *
        np.pi *
        carrier_frequency *
        t
    )

    envelope = np.abs(
        lowpass(
            np.random.randn(n)
            .astype(np.float32),
            12
        )
    )

    envelope /= (
        np.max(envelope)
        + 1e-8
    )

    return (
        0.65 *
        background
        +
        0.35 *
        carrier *
        envelope
    ).astype(
        np.float32
    )


def hum_noise(n):

    t = np.arange(
        n
    ) / SR

    f = random.choice(
        [50.0, 60.0]
    )

    y = np.sin(
        2*np.pi*f*t
    )

    y += (
        0.35 *
        np.sin(
            2*np.pi*2*f*t
        )
    )

    y += (
        0.18 *
        np.sin(
            2*np.pi*3*f*t
        )
    )

    y += (
        0.05 *
        np.random.randn(n)
    )

    return y.astype(
        np.float32
    )


def babble_noise(n):

    y = np.zeros(
        n,
        dtype=np.float32
    )

    t = np.arange(
        n
    ) / SR

    speakers = np.random.randint(
        4,
        9
    )

    for _ in range(
        speakers
    ):

        envelope = np.abs(
            lowpass(
                np.random.randn(n)
                .astype(np.float32),
                np.random.uniform(
                    2,
                    8
                )
            )
        )

        envelope /= (
            np.max(envelope)
            + 1e-8
        )

        f1 = np.random.uniform(
            100,
            350
        )

        f2 = np.random.uniform(
            350,
            1800
        )

        y += envelope * (
            np.sin(
                2*np.pi*f1*t
            )
            +
            0.4 *
            np.sin(
                2*np.pi*f2*t
            )
        )

    return y.astype(
        np.float32
    )


def impulse_noise(n):

    y = np.zeros(
        n,
        dtype=np.float32
    )

    count = np.random.randint(
        1,
        5
    )

    for _ in range(
        count
    ):

        center = np.random.randint(
            0,
            n
        )

        width = np.random.randint(
            int(0.0008 * SR),
            int(0.012 * SR)
        )

        start = max(
            0,
            center - width // 2
        )

        end = min(
            n,
            center + width // 2
        )

        length = end - start

        if length <= 0:
            continue

        t = np.arange(
            length
        ) / SR

        envelope = np.exp(
            -np.linspace(
                0,
                7,
                length
            )
        )

        frequency = np.random.uniform(
            700,
            6000
        )

        pulse = (
            envelope *
            np.sin(
                2 *
                np.pi *
                frequency *
                t
            )
        )

        pulse += (
            0.25 *
            envelope *
            np.random.randn(
                length
            )
        )

        y[
            start:end
        ] += (
            np.random.uniform(
                0.7,
                2.0
            ) *
            pulse
        )

    return y.astype(
        np.float32
    )


def generate_noise(n):

    noise_type = random.choices(

        [
            "engine",
            "rotor",
            "wind",
            "radio",
            "hum",
            "babble",
            "impulse",
            "pink",
            "white",
            "mixed"
        ],

        weights=[
            14,
            12,
            10,
            12,
            8,
            10,
            14,
            6,
            4,
            10
        ],

        k=1
    )[0]


    if noise_type == "engine":

        noise = engine_noise(n)

    elif noise_type == "rotor":

        noise = rotor_noise(n)

    elif noise_type == "wind":

        noise = wind_noise(n)

    elif noise_type == "radio":

        noise = radio_noise(n)

    elif noise_type == "hum":

        noise = hum_noise(n)

    elif noise_type == "babble":

        noise = babble_noise(n)

    elif noise_type == "impulse":

        noise = impulse_noise(n)

    elif noise_type == "pink":

        noise = pink_noise(n)

    elif noise_type == "white":

        noise = white_noise(n)

    else:

        n1, _ = generate_noise(n)
        n2, _ = generate_noise(n)

        noise = (
            0.65*n1 +
            0.45*n2
        )

        noise_type = "mixed"


    return (
        noise.astype(np.float32),
        noise_type
    )


# ================================================================
# 9. FIND DATASET FILES
# ================================================================

def find_wavs(folder):

    if not os.path.isdir(folder):

        return []

    return sorted(
        str(p)
        for p in Path(folder).rglob(
            "*.wav"
        )
    )


clean_files = []
noise_files = []


# Supplied AMPORA speech
clean_files += find_wavs(
    os.path.join(
        DATASET_ROOT,
        "clean_speech"
    )
)


# Supplied noise
noise_files += find_wavs(
    os.path.join(
        DATASET_ROOT,
        "noise_original"
    )

)

noise_files += find_wavs(
    os.path.join(
        DATASET_ROOT,
        "noise_impulsive"
    )
)


# FSDD real speech
if FSDD_ROOT_ACTUAL:

    clean_files += find_wavs(
        os.path.join(
            FSDD_ROOT_ACTUAL,
            "recordings"
        )
    )


clean_files = sorted(
    set(clean_files)
)

noise_files = sorted(
    set(noise_files)
)


print(
    "Clean speech:",
    len(clean_files)
)

print(
    "External noise:",
    len(noise_files)
)


# ================================================================
# 10. SPEAKER-DISJOINT SPLIT
# ================================================================

def speaker_id(path):

    name = Path(
        path
    ).stem.lower()

    # FSDD:
    # digit_speaker_index.wav

    parts = name.split("_")

    if (
        len(parts) >= 2
        and parts[0].isdigit()
    ):

        return (
            "fsdd_" +
            parts[1]
        )

    # Try speaker labels

    match = re.search(
        r"(?:speaker|spk|person|voice)[-_]?([a-z0-9]+)",
        name
    )

    if match:

        return (
            "speaker_" +
            match.group(1)
        )

    # Unknown speech file:
    # treat each file as its own group

    return (
        "file_" +
        name
    )


def speaker_split(
    files,
    validation_fraction=0.20
):

    groups = defaultdict(list)

    for f in files:

        groups[
            speaker_id(f)
        ].append(f)

    speakers = list(
        groups.keys()
    )

    random.Random(
        SEED
    ).shuffle(
        speakers
    )

    if len(speakers) <= 1:

        return files, []

    val_count = max(
        1,
        int(
            round(
                len(speakers) *
                validation_fraction
            )
        )
    )

    val_count = min(
        val_count,
        len(speakers)-1
    )

    validation_speakers = set(
        speakers[:val_count]
    )

    train = []
    validation = []

    for spk, items in groups.items():

        if spk in validation_speakers:

            validation.extend(
                items
            )

        else:

            train.extend(
                items
            )

    return (
        sorted(train),
        sorted(validation)
    )


train_clean, val_clean = speaker_split(
    clean_files
)


print(
    "Training speech:",
    len(train_clean)
)

print(
    "Validation speech:",
    len(val_clean)
)

print(
    "Validation speakers:",
    sorted(
        set(
            speaker_id(x)
            for x in val_clean
        )
    )
)



# ================================================================
# 10A. SMALL REAL-WORLD AUGMENTATIONS
# ================================================================
# These are deliberately simple: they make the training data less "clean"
# without turning the notebook into a giant preprocessing framework.

def random_reverb(x):
    """Add a short synthetic room/vehicle-style tail."""
    if np.random.rand() > 0.35:
        return x

    rir_len = np.random.randint(int(0.03 * SR), int(0.18 * SR))
    t = np.arange(rir_len, dtype=np.float32) / SR
    decay = np.exp(-t / np.random.uniform(0.02, 0.08)).astype(np.float32)

    rir = decay * np.random.randn(rir_len).astype(np.float32)
    rir[0] += 1.0

    rir /= np.sqrt(np.sum(rir * rir) + 1e-8)
    y = signal.fftconvolve(x, rir, mode="full")[:len(x)]

    return normalize_peak(y, peak=0.95)


def random_clip(x):
    """Occasional mild ADC/communication clipping."""
    if np.random.rand() > 0.20:
        return x

    limit = np.random.uniform(0.55, 0.90)
    y = np.clip(x, -limit, limit)

    # Put the signal back into a reasonable training range.
    return (y / (limit + 1e-8)).astype(np.float32)


def augment_clean(x):
    x = random_reverb(x)
    x = random_clip(x)

    # Small gain change keeps the network from learning one fixed level.
    x *= np.random.uniform(0.85, 1.05)
    return np.clip(x, -1.0, 1.0).astype(np.float32)


# ================================================================
# 11. HIGH-BAND DATASET
# ================================================================

class HighBandDataset(
    Dataset
):

    def __init__(
        self,
        clean_paths,
        noise_paths,
        training=True
    ):

        self.clean_paths = clean_paths

        self.noise_paths = noise_paths

        self.training = training

        self.epoch = 0

        self.snr_low = (
            SNR_MIN
        )

        self.snr_high = (
            SNR_MAX
        )


    def set_epoch(
        self,
        epoch
    ):

        self.epoch = epoch


    def __len__(self):

        if self.training:

            return max(
                1200,
                len(
                    self.clean_paths
                ) * 12
            )

        return min(
            300,
            max(
                1,
                len(
                    self.clean_paths
                ) * 4
            )
        )


    def __getitem__(
        self,
        index
    ):

        # Deterministic worker seed
        seed = (
            SEED
            +
            index * 1009
            +
            self.epoch * 100003
        )

        random.seed(seed)

        np.random.seed(
            seed %
            (2**32 - 1)
        )


        # --------------------------------------------------------
        # CLEAN SPEECH
        # --------------------------------------------------------

        clean_path = (
            self.clean_paths[
                index %
                len(
                    self.clean_paths
                )
            ]
        )

        clean = read_audio(
            clean_path
        )

        clean = fit_length(
            clean,
            SEGMENT_SAMPLES
        )

        clean = (
            clean /
            (rms(clean)+1e-8)
        )

        clean *= np.random.uniform(
            0.25,
            0.8
        )

        # A little room response + clipping makes the training examples
        # closer to what a real communication microphone can produce.
        if self.training:
            clean = augment_clean(clean)


        # --------------------------------------------------------
        # NOISE
        # --------------------------------------------------------

        use_external = (
            len(self.noise_paths) > 0
            and
            np.random.rand() < 0.25
        )

        if use_external:

            noise_path = (
                self.noise_paths[
                    np.random.randint(
                        len(
                            self.noise_paths
                        )
                    )
                ]
            )

            noise = read_audio(
                noise_path
            )

            noise = fit_length(
                noise,
                SEGMENT_SAMPLES
            )

        else:

            noise, _ = generate_noise(
                SEGMENT_SAMPLES
            )


        # Additional disturbance
        if np.random.rand() < 0.30:

            n2, _ = generate_noise(
                SEGMENT_SAMPLES
            )

            noise += (
                np.random.uniform(
                    0.2,
                    0.6
                )
                *
                n2
            )


        # --------------------------------------------------------
        # RANDOM SNR
        # --------------------------------------------------------

        snr = np.random.uniform(
            self.snr_low,
            self.snr_high
        )

        noise = scale_noise_to_snr(
            clean,
            noise,
            snr
        )

        noisy = (
            clean +
            noise
        )

        noisy = normalize_peak(
            noisy
        )


        # --------------------------------------------------------
        # HIGH-BAND EXTRACTION
        # --------------------------------------------------------
        #
        # THIS IS THE CRITICAL PART.
        #
        # The neural network receives ONLY:
        #
        #                  1.5 - 8 kHz
        #
        # No low-band signal is passed to it.
        #
        # --------------------------------------------------------

        clean_high = extract_high_band(
            clean
        )

        noisy_high = extract_high_band(
            noisy
        )


        # Small input gain variation
        if self.training:

            noisy_high *= (
                np.random.uniform(
                    0.90,
                    1.10
                )
            )


        return (

            torch.from_numpy(
                noisy_high
            ).float().unsqueeze(0),

            torch.from_numpy(
                clean_high
            ).float().unsqueeze(0)

        )


# ================================================================
# 12. DATA LOADERS
# ================================================================

train_dataset = HighBandDataset(
    train_clean,
    noise_files,
    training=True
)

val_dataset = HighBandDataset(
    val_clean if val_clean else train_clean[:1],
    noise_files,
    training=False
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

# Fast pre-training sanity check. num_workers=0 keeps the active function
# definitions in the same Colab process and avoids stale worker functions.
_test_noisy, _test_clean = next(iter(train_loader))
print("Sanity check OK")
print("  noisy batch shape:", tuple(_test_noisy.shape))
print("  clean batch shape:", tuple(_test_clean.shape))


# ================================================================
# 13. STFT
# ================================================================

WINDOW = torch.hann_window(
    WIN_LENGTH
)


def stft_audio(x):

    if x.ndim == 3:

        x = x[:, 0]

    return torch.stft(

        x,

        n_fft=NFFT,

        hop_length=HOP_LENGTH,

        win_length=WIN_LENGTH,

        window=WINDOW.to(
            x.device
        ),

        return_complex=True,

        center=True,

        pad_mode="reflect"
    )


def istft_audio(
    X,
    length
):

    return torch.istft(

        X,

        n_fft=NFFT,

        hop_length=HOP_LENGTH,

        win_length=WIN_LENGTH,

        window=WINDOW.to(
            X.device
        ),

        length=length,

        center=True
    )


# ================================================================
# 14. HIGH-BAND STFT BIN SELECTION
# ================================================================
#
# DC-CRN receives only the bins corresponding to:
#
#             1.5 kHz - 8 kHz
#
# ================================================================

frequency_resolution = (
    SR / NFFT
)

# ================================================================
# AMPORA HIGH-BAND CONFIGURATION
# ================================================================

SR = 16000

# AI/ML HIGH-BAND:
# 1.5 kHz to 8 kHz
HIGH_BAND_LOW_HZ = 1500.0
HIGH_BAND_HIGH_HZ = 8000.0

# STFT
N_FFT = 512
WIN_LEN = 320
HOP = 128

# Frequency resolution
frequency_resolution = SR / N_FFT

# STFT frequency-bin limits
HIGH_BIN_START = int(
    np.ceil(
        HIGH_BAND_LOW_HZ /
        frequency_resolution
    )
)

HIGH_BIN_END = min(
    N_FFT // 2 + 1,
    int(np.floor(HIGH_BAND_HIGH_HZ / frequency_resolution)) + 1
)

print("==============================================")
print("HIGH-BAND CONFIGURATION")
print("==============================================")
print(f"Sampling rate       : {SR} Hz")
print(f"High-band           : {HIGH_BAND_LOW_HZ/1000:.1f}–{HIGH_BAND_HIGH_HZ/1000:.1f} kHz")
print(f"Frequency resolution : {frequency_resolution:.2f} Hz")
print(f"HIGH_BIN_START      : {HIGH_BIN_START}")
print(f"HIGH_BIN_END        : {HIGH_BIN_END}")
print("==============================================")

# ================================================================
# 15. DC-CRN
# ================================================================

class ConvBlock(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=(1,1)
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=stride,
                padding=1
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.PReLU(
                out_channels
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.PReLU(
                out_channels
            )
        )


    def forward(
        self,
        x
    ):

        return self.block(x)


class DecoderBlock(
    nn.Module
):

    def __init__(
        self,
        input_channels,
        skip_channels,
        output_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                input_channels +
                skip_channels,
                output_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                output_channels
            ),

            nn.PReLU(
                output_channels
            ),

            nn.Conv2d(
                output_channels,
                output_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                output_channels
            ),

            nn.PReLU(
                output_channels
            )
        )


    def forward(
        self,
        x,
        skip
    ):

        x = F.interpolate(

            x,

            size=(
                skip.shape[-2],
                x.shape[-1]
            ),

            mode="bilinear",

            align_corners=False
        )

        x = torch.cat(
            [
                x,
                skip
            ],
            dim=1
        )

        return self.block(x)


class AMPORA_DCCRN(
    nn.Module
):

    def __init__(
        self,
        base=16,
        gru_hidden=64
    ):

        super().__init__()


        # --------------------------------------------------------
        # ENCODER
        # --------------------------------------------------------

        self.enc1 = ConvBlock(
            2,
            base
        )

        self.enc2 = ConvBlock(
            base,
            base*2,
            stride=(2,1)
        )

        self.enc3 = ConvBlock(
            base*2,
            base*3,
            stride=(2,1)
        )

        self.enc4 = ConvBlock(
            base*3,
            base*4,
            stride=(2,1)
        )


        # --------------------------------------------------------
        # TEMPORAL RECURRENT PART
        # --------------------------------------------------------

        self.gru = nn.GRU(

            input_size=base*4,

            hidden_size=gru_hidden,

            num_layers=1,

            batch_first=True,

            bidirectional=False
        )


        self.gru_projection = nn.Linear(

            gru_hidden,

            base * 4
        )


        # --------------------------------------------------------
        # DECODER
        # --------------------------------------------------------

        self.dec3 = DecoderBlock(
            base*4,
            base*3,
            base*3
        )

        self.dec2 = DecoderBlock(
            base*3,
            base*2,
            base*2
        )

        self.dec1 = DecoderBlock(
            base*2,
            base,
            base
        )


        # Complex mask:
        #
        # channel 0 = real
        # channel 1 = imaginary

        self.output = nn.Conv2d(
            base,
            2,
            kernel_size=1
        )


    def forward(
        self,
        x
    ):

        # x:
        # [B, 2, F, T]

        e1 = self.enc1(x)

        e2 = self.enc2(e1)

        e3 = self.enc3(e2)

        e4 = self.enc4(e3)


        # --------------------------------------------------------
        # GRU
        # --------------------------------------------------------

        # Frequency pooling
        sequence = e4.mean(
            dim=2
        )

        # [B,C,T] -> [B,T,C]
        sequence = sequence.transpose(
            1,
            2
        )

        sequence, _ = self.gru(
            sequence
        )

        sequence = self.gru_projection(
            sequence
        )

        # [B,T,C] -> [B,C,1,T]

        context = sequence.transpose(
            1,
            2
        ).unsqueeze(2)


        e4 = e4 + context


        # --------------------------------------------------------
        # DECODER
        # --------------------------------------------------------

        d3 = self.dec3(
            e4,
            e3
        )

        d2 = self.dec2(
            d3,
            e2
        )

        d1 = self.dec1(
            d2,
            e1
        )


        d1 = F.interpolate(

            d1,

            size=(
                x.shape[-2],
                x.shape[-1]
            ),

            mode="bilinear",

            align_corners=False
        )


        # Bounded complex residual mask

        mask = torch.tanh(
            self.output(d1)
        )


        return mask


# ================================================================
# 16. CREATE MODEL
# ================================================================

model = AMPORA_DCCRN(
    base=16,
    gru_hidden=64
).to(
    DEVICE
)


parameters = sum(
    p.numel()
    for p in model.parameters()
)


print(
    f"Model parameters: "
    f"{parameters/1e6:.2f} M"
)

# ================================================================
# 16A. MODEL FORWARD SANITY TEST
# ================================================================
# This confirms the model dimensions BEFORE any training starts.
# The GRU receives base*4 channels after frequency pooling, so
# input_size = 16*4 = 64.
# ================================================================

model.eval()

with torch.no_grad():
    _dummy_input = torch.randn(
        2,
        2,
        HIGH_BIN_END - HIGH_BIN_START,
        161,
        device=DEVICE
    )

    _dummy_output = model(
        _dummy_input
    )

assert _dummy_output.shape == _dummy_input.shape, (
    f"Model forward shape mismatch: "
    f"input={tuple(_dummy_input.shape)}, "
    f"output={tuple(_dummy_output.shape)}"
)

print("MODEL FORWARD TEST: PASSED")
print("  input :", tuple(_dummy_input.shape))
print("  output:", tuple(_dummy_output.shape))
print("  GRU input_size:", model.gru.input_size)

model.train()



# ================================================================
# 17. COMPLEX NUMBER UTILITIES
# ================================================================

def complex_to_channels(
    z
):

    return torch.stack(
        [
            z.real,
            z.imag
        ],
        dim=1
    )


def channels_to_complex(
    x
):

    return torch.complex(
        x[:,0],
        x[:,1]
    )


def apply_complex_mask(
    noisy_complex,
    mask
):

    complex_mask = (
        channels_to_complex(
            mask
        )
    )

    # Residual complex mask
    #
    # Y = X * (1 + M)

    enhanced = (
        noisy_complex *
        (1.0 + complex_mask)
    )

    return enhanced


# ================================================================
# 18. HIGH-BAND MODEL FORWARD
# ================================================================

def enhance_batch(
    noisy_high
):

    length = (
        noisy_high.shape[-1]
    )


    # Full STFT
    X_full = stft_audio(
        noisy_high
    )


    # ------------------------------------------------------------
    # IMPORTANT:
    #
    # ONLY HIGH BAND IS PASSED TO DC-CRN.
    #
    # 1.5 kHz - 8 kHz
    # ------------------------------------------------------------

    X_high = X_full[
        :,
        HIGH_BIN_START:
        HIGH_BIN_END,
        :
    ]


    # Convert complex STFT
    # to real/imag channels

    X_input = complex_to_channels(
        X_high
    )


    # ------------------------------------------------------------
    # DC-CRN
    # ------------------------------------------------------------

    mask = model(
        X_input
    )


    # Enhanced high-band spectrum

    Y_high = apply_complex_mask(
        X_high,
        mask
    )


    # ------------------------------------------------------------
    # Put enhanced HIGH BAND back
    # into full STFT.
    #
    # Everything outside 1.5-8 kHz = zero.
    # ------------------------------------------------------------

    Y_full = torch.zeros_like(
        X_full
    )


    Y_full[
        :,
        HIGH_BIN_START:
        HIGH_BIN_END,
        :
    ] = Y_high


    # Inverse STFT

    enhanced = istft_audio(
        Y_full,
        length
    )


    return enhanced.unsqueeze(1)


# ================================================================
# 19. LOSSES
# ================================================================

def si_sdr(
    estimate,
    target,
    eps=1e-8
):

    estimate = (
        estimate -
        estimate.mean(
            dim=-1,
            keepdim=True
        )
    )

    target = (
        target -
        target.mean(
            dim=-1,
            keepdim=True
        )
    )


    projection = (

        torch.sum(
            estimate *
            target,
            dim=-1,
            keepdim=True
        )

        /

        (
            torch.sum(
                target *
                target,
                dim=-1,
                keepdim=True
            )
            +
            eps
        )

    ) * target


    error = (
        estimate -
        projection
    )


    ratio = (

        torch.sum(
            projection *
            projection,
            dim=-1
        )

        /

        (
            torch.sum(
                error *
                error,
                dim=-1
            )
            +
            eps
        )

    )


    return (
        10 *
        torch.log10(
            ratio +
            eps
        )
    )


def si_sdr_loss(
    estimate,
    target
):

    return -si_sdr(
        estimate,
        target
    ).mean()


def multi_resolution_stft_loss(
    estimate,
    target
):

    settings = [

        (256,160,64),

        (512,320,128),

        (1024,640,256)

    ]

    total = 0.0


    for nfft, win, hop in settings:

        window = torch.hann_window(
            win,
            device=estimate.device
        )


        E = torch.stft(

            estimate.squeeze(1),

            n_fft=nfft,

            hop_length=hop,

            win_length=win,

            window=window,

            return_complex=True,

            center=True,

            pad_mode="reflect"
        )


        T = torch.stft(

            target.squeeze(1),

            n_fft=nfft,

            hop_length=hop,

            win_length=win,

            window=window,

            return_complex=True,

            center=True,

            pad_mode="reflect"
        )


        E_mag = torch.abs(E)

        T_mag = torch.abs(T)


        log_loss = F.l1_loss(

            torch.log1p(E_mag),

            torch.log1p(T_mag)
        )


        spectral_error = (

            torch.linalg.vector_norm(
                T_mag -
                E_mag,
                dim=(-2,-1)
            )

            /

            (
                torch.linalg.vector_norm(
                    T_mag,
                    dim=(-2,-1)
                )
                + 1e-8
            )

        ).mean()


        total += (
            log_loss +
            0.5 *
            spectral_error
        )


    return (
        total /
        len(settings)
    )


def complex_loss(
    estimate,
    target
):

    E = stft_audio(
        estimate
    )

    T = stft_audio(
        target
    )


    E = E[
        :,
        HIGH_BIN_START:
        HIGH_BIN_END
    ]

    T = T[
        :,
        HIGH_BIN_START:
        HIGH_BIN_END
    ]


    return F.l1_loss(

        torch.view_as_real(E),

        torch.view_as_real(T)
    )


def total_loss(
    estimate,
    target
):

    loss_mrstft = (
        multi_resolution_stft_loss(
            estimate,
            target
        )
    )

    loss_complex = (
        complex_loss(
            estimate,
            target
        )
    )

    loss_sisdr = (
        si_sdr_loss(
            estimate.squeeze(1),
            target.squeeze(1)
        )
    )


    loss = (

        1.0 *
        loss_mrstft

        +

        0.5 *
        loss_complex

        +

        0.15 *
        loss_sisdr
    )


    return loss


# ================================================================
# 20. OPTIMIZER
# ================================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    betas=(0.9,0.98)
)


scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(

        optimizer,

        T_max=EPOCHS,

        eta_min=2e-6
    )
)


scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())


# ================================================================
# 21. TRAINING
# ================================================================

def train_one_epoch(
    epoch
):

    model.train()

    train_dataset.set_epoch(
        epoch
    )


    # Curriculum
    if epoch < 5:

        train_dataset.snr_low = -5
        train_dataset.snr_high = 10

    elif epoch < 10:

        train_dataset.snr_low = -10
        train_dataset.snr_high = 10

    else:

        train_dataset.snr_low = -15
        train_dataset.snr_high = 10


    running_loss = 0.0


    progress = tqdm(

        train_loader,

        desc=
        f"Epoch "
        f"{epoch+1}/{EPOCHS}"
    )


    for noisy_high, clean_high in progress:

        noisy_high = (
            noisy_high.to(
                DEVICE,
                non_blocking=True
            )
        )

        clean_high = (
            clean_high.to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(

            device_type="cuda",
            enabled=torch.cuda.is_available()

        ):

            enhanced_high = (
                enhance_batch(
                    noisy_high
                )
            )


            loss = total_loss(

                enhanced_high,

                clean_high
            )


        scaler.scale(
            loss
        ).backward()


        scaler.unscale_(
            optimizer
        )


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            GRAD_CLIP
        )


        scaler.step(
            optimizer
        )

        scaler.update()


        running_loss += (
            loss.item()
        )


        progress.set_postfix(

            loss=
            f"{running_loss / max(1, progress.n + 1):.4f}",

            lr=
            f"{optimizer.param_groups[0]['lr']:.2e}",

            snr=
            f"{train_dataset.snr_low}"
            f".."
            f"{train_dataset.snr_high}"
        )


    return (
        running_loss /
        len(train_loader)
    )


# ================================================================
# 22. VALIDATION
# ================================================================

@torch.no_grad()
def validate():

    model.eval()

    total_loss_value = 0.0

    total_improvement = []


    for noisy_high, clean_high in tqdm(

        val_loader,

        desc="Validation"

    ):

        noisy_high = noisy_high.to(
            DEVICE
        )

        clean_high = clean_high.to(
            DEVICE
        )


        enhanced_high = (
            enhance_batch(
                noisy_high
            )
        )


        loss = total_loss(

            enhanced_high,

            clean_high
        )


        total_loss_value += (
            loss.item()
        )


        noisy_sisdr = si_sdr(

            noisy_high.squeeze(1),

            clean_high.squeeze(1)
        )


        enhanced_sisdr = si_sdr(

            enhanced_high.squeeze(1),

            clean_high.squeeze(1)
        )


        improvement = (
            enhanced_sisdr -
            noisy_sisdr
        )


        total_improvement.extend(

            improvement
            .cpu()
            .numpy()
            .tolist()
        )


    return (

        total_loss_value /
        len(val_loader),

        np.mean(
            total_improvement
        )
    )


# ================================================================
# 23. CHECKPOINTS + RESUME SUPPORT
# ================================================================

BEST_MODEL = os.path.join(
    CHECKPOINT_DIR,
    "AMPORA_HIGHBAND_DCCRN_BEST.pt"
)

LAST_MODEL = os.path.join(
    CHECKPOINT_DIR,
    "AMPORA_HIGHBAND_DCCRN_LAST.pt"
)

HISTORY_FILE = os.path.join(
    CHECKPOINT_DIR,
    "training_history.csv"
)


def save_checkpoint(
    path,
    epoch,
    best_loss,
    history_data
):

    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_loss": best_loss,
            "history": history_data,
            "config": {
                "sample_rate": SR,
                "global_band": [GLOBAL_LOW, GLOBAL_HIGH],
                "high_band": [CROSSOVER, HIGH_BAND_HIGH_HZ],
                "nfft": NFFT,
                "win_length": WIN_LENGTH,
                "hop_length": HOP_LENGTH,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE
            }
        },
        path
    )


def load_training_checkpoint(path):

    checkpoint = torch.load(
    BEST_MODEL,
    map_location=DEVICE,
    weights_only=False
)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])

    if "scaler" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler"])

    loaded_history = checkpoint.get("history", [])
    loaded_best = checkpoint.get("best_loss", float("inf"))
    loaded_epoch = checkpoint.get("epoch", -1)

    return loaded_epoch, loaded_best, loaded_history


# ================================================================
# 24. MAIN TRAINING LOOP — DRIVE SAFE / AUTO RESUME
# ================================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is currently available. Do not train on CPU. "
        "Reconnect a Colab GPU and rerun this notebook. "
        f"Your Drive checkpoints will remain in: {CHECKPOINT_DIR}"
    )

start_epoch = 0
best_validation_loss = float("inf")
history = []

# Automatically resume from the last completed epoch if a checkpoint exists.
if os.path.exists(LAST_MODEL):

    print("\nRESUME CHECKPOINT FOUND")
    print(LAST_MODEL)

    loaded_epoch, best_validation_loss, history = load_training_checkpoint(
        LAST_MODEL
    )

    start_epoch = loaded_epoch + 1

    print("Last completed epoch:", loaded_epoch + 1)
    print("Next epoch:", start_epoch + 1)
    print("Best validation loss:", best_validation_loss)

else:

    print("\nNo previous checkpoint found.")
    print("Starting training from epoch 1.")


if start_epoch >= EPOCHS:

    print("\nTRAINING ALREADY COMPLETED.")
    print("No additional epochs are required.")

else:

    for epoch in range(
        start_epoch,
        EPOCHS
    ):

        train_loss = train_one_epoch(
            epoch
        )

        validation_loss, delta_sisdr = validate()

        scheduler.step()

        print(
            "\n--------------------------------"
        )
        print(
            "Epoch:",
            epoch + 1
        )
        print(
            "Train loss:",
            train_loss
        )
        print(
            "Validation loss:",
            validation_loss
        )
        print(
            "Validation ΔSI-SDR:",
            delta_sisdr,
            "dB"
        )
        print(
            "--------------------------------"
        )

        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "validation_loss": validation_loss,
                "delta_si_sdr": delta_sisdr,
                "learning_rate": optimizer.param_groups[0]["lr"]
            }
        )

        # Update best BEFORE saving so the checkpoint contains the new best value.
        if validation_loss < best_validation_loss:

            best_validation_loss = validation_loss

            save_checkpoint(
                BEST_MODEL,
                epoch,
                best_validation_loss,
                history
            )

            print("NEW BEST MODEL SAVED")

        # ALWAYS save the latest completed epoch.
        save_checkpoint(
            LAST_MODEL,
            epoch,
            best_validation_loss,
            history
        )

        # CSV history is also stored in Drive.
        pd.DataFrame(history).to_csv(
            HISTORY_FILE,
            index=False
        )

        print("CHECKPOINT SAVED TO GOOGLE DRIVE")
        print("Latest:", LAST_MODEL)
        print("Best:", BEST_MODEL)


print("\nTRAINING SECTION FINISHED")
print("Best model:", BEST_MODEL)
print("Latest checkpoint:", LAST_MODEL)
print("History:", HISTORY_FILE)


# ================================================================
# 25. LOAD BEST MODEL
# ================================================================

checkpoint = torch.load(

    BEST_MODEL,

    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model"]
)

model.eval()


print(
    "Loaded best model from epoch:",
    checkpoint["epoch"] + 1
)


# ================================================================
# 26. NUMPY SI-SDR / SNR
# ================================================================

def numpy_snr(
    clean,
    estimate
):

    error = (
        clean -
        estimate
    )

    return (

        10 *
        np.log10(

            (
                np.sum(
                    clean ** 2
                )
                + 1e-12
            )

            /

            (
                np.sum(
                    error ** 2
                )
                + 1e-12
            )

        )

    )


def numpy_si_sdr(
    estimate,
    target
):

    estimate = (
        estimate -
        np.mean(estimate)
    )

    target = (
        target -
        np.mean(target)
    )


    alpha = (

        np.sum(
            estimate *
            target
        )

        /

        (
            np.sum(
                target *
                target
            )
            + 1e-8
        )

    )


    target_projection = (
        alpha *
        target
    )


    noise = (
        estimate -
        target_projection
    )


    return (

        10 *
        np.log10(

            (
                np.sum(
                    target_projection ** 2
                )
                + 1e-8
            )

            /

            (
                np.sum(
                    noise ** 2
                )
                + 1e-8
            )

        )

    )


# ================================================================
# 27. SINGLE HIGH-BAND INFERENCE
# ================================================================

@torch.no_grad()
def enhance_high_band(
    noisy_high
):

    model.eval()


    x = torch.from_numpy(
        noisy_high
        .astype(np.float32)
    )

    x = x.unsqueeze(
        0
    ).unsqueeze(
        0
    )

    x = x.to(
        DEVICE
    )


    enhanced = enhance_batch(
        x
    )


    return (
        enhanced
        .squeeze()
        .cpu()
        .numpy()
        .astype(np.float32)
    )


# ================================================================
# 28. WAV FILE INFERENCE
# ================================================================

def enhance_wav_high_band(
    input_wav,
    output_wav
):

    print(
        "\nProcessing:",
        input_wav
    )


    # ------------------------------------------------------------
    # Input
    # ------------------------------------------------------------

    audio = read_audio(
        input_wav
    )


    # ------------------------------------------------------------
    # Extract ONLY 1.5-8 kHz
    # ------------------------------------------------------------

    noisy_high = (
        extract_high_band(
            audio
        )
    )


    # ------------------------------------------------------------
    # DC-CRN ONLY
    # ------------------------------------------------------------

    enhanced_high = (
        enhance_high_band(
            noisy_high
        )
    )


    # ------------------------------------------------------------
    # Output is HIGH BAND ONLY
    # ------------------------------------------------------------

    enhanced_high = (
        normalize_peak(
            enhanced_high
        )
    )


    sf.write(

        output_wav,

        enhanced_high,

        SR,

        subtype="PCM_16"
    )


    print(
        "Saved:",
        output_wav
    )


# ================================================================
# 29. EXAMPLE
# ================================================================

# Upload a noisy test WAV to Colab:
#
# /content/test_noisy.wav
#
# Then run:
#
# enhance_wav_high_band(
#
#     "/content/test_noisy.wav",
#
#     "/content/AMPORA_enhanced_HIGHBAND.wav"
#
# )


# ================================================================
# 30. EVALUATION
# ================================================================

try:
    from pystoi import stoi as _stoi
    HAVE_STOI = True
except Exception:
    HAVE_STOI = False

try:
    from pesq import pesq as _pesq
    HAVE_PESQ = True
except Exception:
    HAVE_PESQ = False

print("STOI available:", HAVE_STOI)
print("PESQ available:", HAVE_PESQ)

def evaluate_pair(
    clean_high,
    noisy_high,
    enhanced_high
):

    result = {}


    # SNR

    result[
        "Noisy_SNR_dB"
    ] = numpy_snr(

        clean_high,

        noisy_high
    )


    result[
        "Enhanced_SNR_dB"
    ] = numpy_snr(

        clean_high,

        enhanced_high
    )


    result[
        "Delta_SNR_dB"
    ] = (

        result[
            "Enhanced_SNR_dB"
        ]

        -

        result[
            "Noisy_SNR_dB"
        ]

    )


    # SI-SDR

    result[
        "Noisy_SI_SDR_dB"
    ] = numpy_si_sdr(

        noisy_high,

        clean_high
    )


    result[
        "Enhanced_SI_SDR_dB"
    ] = numpy_si_sdr(

        enhanced_high,

        clean_high
    )


    result[
        "Delta_SI_SDR_dB"
    ] = (

        result[
            "Enhanced_SI_SDR_dB"
        ]

        -

        result[
            "Noisy_SI_SDR_dB"
        ]

    )


    # STOI

    if HAVE_STOI:

        try:

            from pystoi import stoi

            result[
                "Noisy_STOI"
            ] = stoi(

                clean_high,

                noisy_high,

                SR,

                extended=False
            )

            result[
                "Enhanced_STOI"
            ] = stoi(

                clean_high,

                enhanced_high,

                SR,

                extended=False
            )

        except:

            result[
                "Noisy_STOI"
            ] = np.nan

            result[
                "Enhanced_STOI"
            ] = np.nan


    # PESQ

    if HAVE_PESQ:

        try:

            from pesq import pesq

            result[
                "Noisy_PESQ"
            ] = pesq(

                SR,

                clean_high,

                noisy_high,

                "wb"
            )

            result[
                "Enhanced_PESQ"
            ] = pesq(

                SR,

                clean_high,

                enhanced_high,

                "wb"
            )

        except:

            result[
                "Noisy_PESQ"
            ] = np.nan

            result[
                "Enhanced_PESQ"
            ] = np.nan


    return result


# ================================================================
# 31. TEST THE MODEL ON VALIDATION SPEECH
# ================================================================

results = []


for i, path in enumerate(

    tqdm(
        val_clean[
            :min(
                50,
                len(val_clean)
            )
        ],

        desc="Final evaluation"
    )

):


    clean = read_audio(
        path
    )


    clean = fit_length(
        clean,
        SEGMENT_SAMPLES
    )


    # Generate hard test noise

    noise, noise_type = (
        generate_noise(
            len(clean)
        )
    )


    snr = random.choice(

        [
            -15,
            -10,
            -5,
            0,
            5,
            10
        ]

    )


    noise = scale_noise_to_snr(

        clean,

        noise,

        snr
    )


    noisy = normalize_peak(

        clean +
        noise
    )


    # ----------------------------------------
    # HIGH BAND ONLY
    # ----------------------------------------

    clean_high = (
        extract_high_band(
            clean
        )
    )

    noisy_high = (
        extract_high_band(
            noisy
        )
    )


    enhanced_high = (
        enhance_high_band(
            noisy_high
        )
    )


    metrics = evaluate_pair(

        clean_high,

        noisy_high,

        enhanced_high
    )


    metrics[
        "Noise_Type"
    ] = noise_type

    metrics[
        "Test_SNR"
    ] = snr

    metrics[
        "File"
    ] = path


    results.append(
        metrics
    )


results_df = pd.DataFrame(
    results
)


metrics_path = os.path.join(

    CHECKPOINT_DIR,

    "AMPORA_HIGHBAND_METRICS.csv"
)


results_df.to_csv(

    metrics_path,

    index=False
)


print(
    "\n========================================"
)

print(
    "HIGH-BAND AI RESULTS"
)

print(
    "========================================"
)


if len(results_df) > 0:

    print(

        results_df
        .select_dtypes(
            include=np.number
        )
        .mean()
        .round(4)
    )


print(
    "\nMetrics saved to:",
    metrics_path
)


# ================================================================
# 32. SAVE FINAL MODEL
# ================================================================

FINAL_MODEL = os.path.join(

    CHECKPOINT_DIR,

    "AMPORA_HIGHBAND_DC_CRN_FINAL.pt"
)


torch.save(

    {

        "model_state_dict":
        model.state_dict(),

        "sample_rate":
        SR,

        "global_band":
        [
            80,
            8000
        ],

        "ai_band":
        [
            1500,
            8000
        ],

        "nfft":
        NFFT,

        "win_length":
        WIN_LENGTH,

        "hop_length":
        HOP_LENGTH,

        "architecture":
        "AMPORA Lite Causal DC-CRN",

        "quantization":
        "INT8 ONNX controller export",

        "description":
        (
            "AI processes ONLY 1.5-8 kHz. "
            "Low-band 80 Hz-1.5 kHz NLMS "
            "is external to this model."
        )

    },

    FINAL_MODEL
)


print(
    "\nFINAL MODEL:"
)

print(
    FINAL_MODEL
)


# ================================================================
# 33. MODEL CONFIGURATION
# ================================================================

config = {

    "sample_rate":
    SR,

    "global_band_low_hz":
    80,

    "global_band_high_hz":
    8000,

    "ai_band_low_hz":
    1500,

    "ai_band_high_hz":
    8000,

    "nfft":
    NFFT,

    "win_length":
    WIN_LENGTH,

    "hop_length":
    HOP_LENGTH,

    "epochs":
    EPOCHS,

    "batch_size":
    BATCH_SIZE,

    "learning_rate":
    LEARNING_RATE,

    "training_snr_db":
    [
        -15,
        10
    ],

    "model":
    "DC-CRN",

    "low_band_model":
    "External NLMS - NOT INCLUDED",

    "high_band_model":
    "DC-CRN - INCLUDED"

}


with open(

    os.path.join(
        CHECKPOINT_DIR,
        "AMPORA_HIGHBAND_CONFIG.json"
    ),

    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )


print(
    "\n========================================"
)

print(
    "AMPORA HIGH-BAND AI PIPELINE COMPLETE"
)

print(
    "========================================"

)

print(
    """
AI INPUT:
    1.5 kHz - 8 kHz noisy audio

AI MODEL:
    DC-CRN

AI OUTPUT:
    1.5 kHz - 8 kHz enhanced audio

NOT INCLUDED:
    80 Hz - 1.5 kHz NLMS
"""
)

Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints
Device: cuda
GPU: Tesla T4
UPLOAD AMPORA V3 DATASET ZIP
Select your AMPORA_V3_HARD_REALVOICE_DATASET.zip file.
The code automatically detects the ZIP filename.



Saving AMPORA_V3_HARD_REALVOICE_DATASET.zip to AMPORA_V3_HARD_REALVOICE_DATASET.zip
Dataset ZIP detected: /content/AMPORA_V3_HARD_REALVOICE_DATASET.zip
Extracting dataset...
Extraction completed. WAV files found: 2345
FSDD downloaded.
Clean speech: 3025
External noise: 120
Training speech: 2520
Validation speech: 505
Validation speakers: ['fsdd_lucas', 'speaker_006', 'speaker_010', 'speaker_014', 'speaker_019', 'speaker_024']
Training batches: 3780
Validation batches: 38
Sanity check OK
  noisy batch shape: (8, 1, 20480)
  clean batch shape: (8, 1, 20480)
HIGH-BAND CONFIGURATION
Sampling rate       : 16000 Hz
High-band           : 1.5–8.0 kHz
Frequency resolution : 31.25 Hz
HIGH_BIN_START      : 48
HIGH_BIN_END        : 257
Model parameters: 0.26 M
MODEL FORWARD TEST: PASSED
  input : (2, 2, 209, 161)
  output: (2, 2, 209, 161)
  GRU input_size: 64

No previous checkpoint found.
Starting training from epoch 1.


Epoch 1/20:   0%|          | 0/3780 [00:00<?, ?it/s]

/tmp/ipykernel_1115/3811692919.py:2184: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at /pytorch/aten/src/ATen/EmptyTensor.cpp:54.)
  return torch.complex(


Validation:   0%|          | 0/38 [00:00<?, ?it/s]

...


--------------------------------
Epoch: 19
Train loss: 0.8297706785971526
Validation loss: 1.0639934790761847
Validation ΔSI-SDR: -1.4990828559796014 dB
--------------------------------
CHECKPOINT SAVED TO GOOGLE DRIVE
Latest: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_LAST.pt
Best: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_BEST.pt


Epoch 20/20:   0%|          | 0/3780 [00:00<?, ?it/s]

Validation:   0%|          | 0/38 [00:00<?, ?it/s]


--------------------------------
Epoch: 20
Train loss: 0.8288994314336272
Validation loss: 1.06256418322262
Validation ΔSI-SDR: -1.4941300976276397 dB
--------------------------------
CHECKPOINT SAVED TO GOOGLE DRIVE
Latest: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_LAST.pt
Best: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_BEST.pt

TRAINING SECTION FINISHED
Best model: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_BEST.pt
Latest checkpoint: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_LAST.pt
History: /content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/training_history.csv


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [3]:
# ================================================================
# AMPORA POST-TRAINING: LOAD BEST -> EVALUATE -> ONNX -> INT8
# ================================================================

import os
import json
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------

BEST_MODEL = "/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_BEST.pt"
CHECKPOINT_DIR = "/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints"

EXPORT_DIR = "/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

FP32_ONNX = os.path.join(
    EXPORT_DIR,
    "ampora_lite_dccrn_fp32.onnx"
)

INT8_ONNX = os.path.join(
    EXPORT_DIR,
    "ampora_lite_dccrn_int8.onnx"
)

# ------------------------------------------------
# 2. Load BEST trained checkpoint
# ------------------------------------------------

print("Loading BEST trained model...")

checkpoint = torch.load(
    BEST_MODEL,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.to(DEVICE)
model.eval()

print("BEST MODEL LOADED")
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Best validation loss:", checkpoint.get("best_loss"))

# ------------------------------------------------
# 3. Final validation
# ------------------------------------------------

print("\nRunning validation...")

val_loss, val_delta_sisdr = validate()

print("\n========================================")
print("FINAL VALIDATION")
print("========================================")
print("Validation loss:", val_loss)
print("Validation ΔSI-SDR:", val_delta_sisdr, "dB")

# ------------------------------------------------
# 4. Export FP32 ONNX
# ------------------------------------------------

print("\nExporting FP32 ONNX...")

dummy_input = torch.randn(
    1,
    2,
    HIGH_BIN_END - HIGH_BIN_START,
    161,
    device=DEVICE
)

with torch.no_grad():
    test_output = model(dummy_input)

print("PyTorch input :", tuple(dummy_input.shape))
print("PyTorch output:", tuple(test_output.shape))

torch.onnx.export(
    model,
    dummy_input,
    FP32_ONNX,
    input_names=["input"],
    output_names=["mask"],
    dynamic_axes={
        "input": {
            0: "batch",
            3: "time"
        },
        "mask": {
            0: "batch",
            3: "time"
        }
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False
)

print("FP32 ONNX SAVED:")
print(FP32_ONNX)

# ------------------------------------------------
# 5. Check ONNX file
# ------------------------------------------------

import onnx

onnx_model = onnx.load(FP32_ONNX)
onnx.checker.check_model(onnx_model)

print("FP32 ONNX CHECK: PASSED")

# ------------------------------------------------
# 6. INT8 quantization
# ------------------------------------------------

print("\nQuantizing FP32 ONNX -> INT8 ONNX...")

from onnxruntime.quantization import (
    quantize_dynamic,
    QuantType
)

quantize_dynamic(
    model_input=FP32_ONNX,
    model_output=INT8_ONNX,
    weight_type=QuantType.QInt8
)

print("INT8 ONNX SAVED:")
print(INT8_ONNX)

# ------------------------------------------------
# 7. Verify both ONNX models
# ------------------------------------------------

import onnxruntime as ort

print("\nVerifying ONNX Runtime inference...")

fp32_session = ort.InferenceSession(
    FP32_ONNX,
    providers=["CPUExecutionProvider"]
)

int8_session = ort.InferenceSession(
    INT8_ONNX,
    providers=["CPUExecutionProvider"]
)

test_np = dummy_input.detach().cpu().numpy()

fp32_output = fp32_session.run(
    ["mask"],
    {"input": test_np}
)[0]

int8_output = int8_session.run(
    ["mask"],
    {"input": test_np}
)[0]

print("FP32 output shape:", fp32_output.shape)
print("INT8 output shape:", int8_output.shape)

print("\n========================================")
print("AMPORA EXPORT COMPLETE")
print("========================================")

print("FP32 ONNX:")
print(FP32_ONNX)

print("\nINT8 ONNX:")
print(INT8_ONNX)

print("\nFP32 size:",
      round(os.path.getsize(FP32_ONNX) / (1024 * 1024), 2),
      "MB")

print("INT8 size:",
      round(os.path.getsize(INT8_ONNX) / (1024 * 1024), 2),
      "MB")

print("\nALL CHECKS PASSED")

Loading BEST trained model...
BEST MODEL LOADED
Checkpoint epoch: 17
Best validation loss: 1.0613255869401128

Running validation...


Validation:   0%|          | 0/38 [00:00<?, ?it/s]


FINAL VALIDATION
Validation loss: 1.0613255869401128
Validation ΔSI-SDR: -1.4822678405046463 dB

Exporting FP32 ONNX...
PyTorch input : (1, 2, 209, 161)
PyTorch output: (1, 2, 209, 161)


/tmp/ipykernel_1115/3015069561.py:85: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4571: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


FP32 ONNX SAVED:
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/exports/ampora_lite_dccrn_fp32.onnx
FP32 ONNX CHECK: PASSED

Quantizing FP32 ONNX -> INT8 ONNX...


INT8 ONNX SAVED:
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/exports/ampora_lite_dccrn_int8.onnx

Verifying ONNX Runtime inference...
FP32 output shape: (1, 2, 209, 161)
INT8 output shape: (1, 2, 209, 161)

AMPORA EXPORT COMPLETE
FP32 ONNX:
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/exports/ampora_lite_dccrn_fp32.onnx

INT8 ONNX:
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/exports/ampora_lite_dccrn_int8.onnx

FP32 size: 0.99 MB
INT8 size: 0.36 MB

ALL CHECKS PASSED


In [4]:
import os

BASE = "/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER"

print("FILES IN PROJECT:")
for root, dirs, files in os.walk(BASE):
    for file in files:
        print(os.path.join(root, file))

FILES IN PROJECT:
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_BEST.pt
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/AMPORA_HIGHBAND_DCCRN_LAST.pt
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/checkpoints/training_history.csv
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/exports/ampora_lite_dccrn_fp32.onnx
/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER/exports/ampora_lite_dccrn_int8.onnx


In [5]:
import os
import shutil

# ==============================
# GITHUB PROJECT FOLDER
# ==============================

DRIVE_BASE = "/content/drive/MyDrive/AMPORA_HIGHBAND_CONTROLLER"
REPO = "/content/AMPORA-Hybrid-ANC"

# Create folders
os.makedirs(REPO, exist_ok=True)
os.makedirs(f"{REPO}/notebooks", exist_ok=True)
os.makedirs(f"{REPO}/models", exist_ok=True)
os.makedirs(f"{REPO}/results", exist_ok=True)
os.makedirs(f"{REPO}/checkpoints", exist_ok=True)

# --------------------------------
# Copy notebook
# --------------------------------
NOTEBOOK = "/content/AMPORA_HIGHBAND_DCCRN_CHECKPOINT_SAFE.ipynb"

if os.path.exists(NOTEBOOK):
    shutil.copy2(
        NOTEBOOK,
        f"{REPO}/notebooks/AMPORA_HIGHBAND_DCCRN.ipynb"
    )
    print("✓ Notebook copied")
else:
    print("⚠ Notebook not found:", NOTEBOOK)

# --------------------------------
# Copy ONNX models
# --------------------------------
FP32 = f"{DRIVE_BASE}/exports/ampora_lite_dccrn_fp32.onnx"
INT8 = f"{DRIVE_BASE}/exports/ampora_lite_dccrn_int8.onnx"

for src, name in [
    (FP32, "ampora_lite_dccrn_fp32.onnx"),
    (INT8, "ampora_lite_dccrn_int8.onnx")
]:
    if os.path.exists(src):
        shutil.copy2(src, f"{REPO}/models/{name}")
        print("✓", name, "copied")
    else:
        print("⚠ Not found:", src)

# --------------------------------
# Copy training history
# --------------------------------
HISTORY = f"{DRIVE_BASE}/checkpoints/training_history.csv"

if os.path.exists(HISTORY):
    shutil.copy2(
        HISTORY,
        f"{REPO}/results/training_history.csv"
    )
    print("✓ Training history copied")
else:
    print("⚠ Training history not found")

# --------------------------------
# Copy controller config if available
# --------------------------------
CONFIG = "/content/AMPORA_HIGHBAND_CONTROLLER_READY/controller_config.json"

if os.path.exists(CONFIG):
    shutil.copy2(CONFIG, f"{REPO}/controller_config.json")
    print("✓ Controller config copied")
else:
    print("⚠ controller_config.json not found")

# --------------------------------
# Show final structure
# --------------------------------
print("\n==============================")
print("GITHUB PROJECT STRUCTURE")
print("==============================")

for root, dirs, files in os.walk(REPO):
    level = root.replace(REPO, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        print(f"{indent}  ├── {file} ({size/1024:.1f} KB)")

⚠ Notebook not found: /content/AMPORA_HIGHBAND_DCCRN_CHECKPOINT_SAFE.ipynb
✓ ampora_lite_dccrn_fp32.onnx copied
✓ ampora_lite_dccrn_int8.onnx copied
✓ Training history copied
⚠ controller_config.json not found

GITHUB PROJECT STRUCTURE
AMPORA-Hybrid-ANC/
  models/
    ├── ampora_lite_dccrn_int8.onnx (369.9 KB)
    ├── ampora_lite_dccrn_fp32.onnx (1017.2 KB)
  checkpoints/
  results/
    ├── training_history.csv (1.7 KB)
  notebooks/


In [6]:
import os

print("SEARCHING FOR NOTEBOOK + CONFIG...\n")

for search_root in ["/content", "/content/drive/MyDrive"]:
    for root, dirs, files in os.walk(search_root):
        for file in files:
            if file.endswith(".ipynb") or file == "controller_config.json":
                print(os.path.join(root, file))

SEARCHING FOR NOTEBOOK + CONFIG...

/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo.ipynb
/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo (1).ipynb
/content/drive/MyDrive/Colab Notebooks/appidiyae zip ah potu (1).ipynb
/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo_error_free (1).ipynb
/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo_FULL_FIXED.ipynb
/content/drive/MyDrive/Colab Notebooks/internal_hackathon.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo.ipynb
/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo (1).ipynb
/content/drive/MyDrive/Colab Notebooks/appidiyae zip ah potu (1).ipynb
/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo_error_free (1).ipynb
/content/drive/MyDrive/Colab Notebooks/tactical_anc_hackathon_demo_FULL_FIXED.ipynb
/content/drive/MyDrive/Colab Notebooks/internal_hackathon.i

In [7]:
import os
import json

folder = "/content/drive/MyDrive/Colab Notebooks"

print("Checking notebooks for AMPORA/DC-CRN code...\n")

for file in os.listdir(folder):
    if not file.endswith(".ipynb"):
        continue

    path = os.path.join(folder, file)

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        text = json.dumps(data)

        if any(x in text for x in [
            "AMPORA_DCCRN",
            "AMPORA_HIGHBAND",
            "DC-CRN",
            "ampora_lite_dccrn",
            "HIGH_BIN_START"
        ]):
            print("✅ MATCH:", file)

    except Exception as e:
        pass

Checking notebooks for AMPORA/DC-CRN code...

✅ MATCH: tactical_anc_hackathon_demo.ipynb
✅ MATCH: tactical_anc_hackathon_demo (1).ipynb
✅ MATCH: appidiyae zip ah potu (1).ipynb
✅ MATCH: internal_hackathon.ipynb
✅ MATCH: Untitled0.ipynb


In [8]:
import os
import json

folder = "/content/drive/MyDrive/Colab Notebooks"

targets = [
    "tactical_anc_hackathon_demo.ipynb",
    "tactical_anc_hackathon_demo (1).ipynb",
    "appidiyae zip ah potu (1).ipynb",
    "internal_hackathon.ipynb",
    "Untitled0.ipynb"
]

print("=" * 70)
print("NOTEBOOK COMPARISON")
print("=" * 70)

for name in targets:
    path = os.path.join(folder, name)

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        text = json.dumps(data)

        print("\n📓", name)
        print("-" * 70)

        checks = {
            "DC-CRN model": "AMPORA_DCCRN" in text,
            "Checkpoint saving": "AMPORA_HIGHBAND_DCCRN_BEST.pt" in text,
            "20 epochs": "EPOCHS = 20" in text or "EPOCHS=20" in text,
            "Google Drive": "MyDrive/AMPORA_HIGHBAND_CONTROLLER" in text,
            "FP32 ONNX": "ampora_lite_dccrn_fp32.onnx" in text,
            "INT8 ONNX": "ampora_lite_dccrn_int8.onnx" in text,
            "Quantization": "quantize_static" in text,
            "ONNX Runtime": "onnxruntime" in text,
        }

        for label, result in checks.items():
            print(f"{'✅' if result else '❌'} {label}")

    except Exception as e:
        print("ERROR:", e)

NOTEBOOK COMPARISON

📓 tactical_anc_hackathon_demo.ipynb
----------------------------------------------------------------------
❌ DC-CRN model
❌ Checkpoint saving
❌ 20 epochs
❌ Google Drive
❌ FP32 ONNX
❌ INT8 ONNX
❌ Quantization
❌ ONNX Runtime

📓 tactical_anc_hackathon_demo (1).ipynb
----------------------------------------------------------------------
❌ DC-CRN model
❌ Checkpoint saving
❌ 20 epochs
❌ Google Drive
❌ FP32 ONNX
❌ INT8 ONNX
❌ Quantization
❌ ONNX Runtime

📓 appidiyae zip ah potu (1).ipynb
----------------------------------------------------------------------
❌ DC-CRN model
❌ Checkpoint saving
❌ 20 epochs
❌ Google Drive
❌ FP32 ONNX
❌ INT8 ONNX
❌ Quantization
❌ ONNX Runtime

📓 internal_hackathon.ipynb
----------------------------------------------------------------------
❌ DC-CRN model
❌ Checkpoint saving
✅ 20 epochs
❌ Google Drive
❌ FP32 ONNX
❌ INT8 ONNX
❌ Quantization
❌ ONNX Runtime

📓 Untitled0.ipynb
----------------------------------------------------------------------
✅